# Hyperparameter Optimization & AutoML

Companion notebook for the [Hyperparameter Optimization lesson](https://ml-viz-ruby.vercel.app/courses/optimization-ml/05-hyperparameter-optimization).

We empirically reproduce the classic result that **random search beats grid search** when only a
few hyperparameters matter, and implement **Successive Halving** to see how early stopping finds a
good config with far less total compute. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## Intuition — searching the settings you can't learn

Weights are learned by gradient descent; **hyperparameters** (learning rate, depth, regularization)
are not — you have to *search* for them by training-and-evaluating. The naive approach, **grid
search**, spends its budget on a lattice and wastes most of it when only a couple of
hyperparameters actually matter. **Random search** samples each dimension at many distinct values,
so it explores the important ones far more finely. Smarter still, **bandit** methods like
**Successive Halving** start many configs cheaply and kill the losers early, and **Bayesian
optimization** models the score surface to pick promising points. We reproduce the classic
"random beats grid" result and implement Successive Halving from scratch.

## 1 — A toy validation surface where only 1 of 2 hyperparameters matters

The score depends strongly on h1 and barely on h2 — the common real-world case. The best h1 is 0.5.

In [ ]:
def val_score(h1, h2):
    return np.exp(-((h1 - 0.5)**2) / 0.02) + 0.03 * np.cos(10 * h2)   # h2 barely matters

best_possible = val_score(0.5, 0.0)
print(f'best achievable score ~ {best_possible:.3f} (at h1=0.5)')

**What to notice:** the toy validation surface depends strongly on `h1` (a sharp Gaussian peak
at 0.5) and barely on `h2` — the common real-world case where only one or two hyperparameters
matter. Finding the good `h1` is the whole game; effort spent resolving `h2` is wasted.

## 2 — Random search vs grid search at equal budget

Both get 16 trials. Grid spends them on a 4×4 lattice (only 4 distinct h1 values). Random tries 16
distinct h1 values — so it samples the dimension that matters far more finely.

In [ ]:
budget = 16
g = np.linspace(0, 1, 4)
grid_best = max(val_score(a, b) for a in g for b in g)

rand_scores = []
for trial in range(200):
    r = np.random.default_rng(trial)
    rand_best = max(val_score(r.random(), r.random()) for _ in range(budget))
    rand_scores.append(rand_best)

print(f'grid search best   (4x4):     {grid_best:.3f}')
print(f'random search best (avg of 200 runs): {np.mean(rand_scores):.3f}')
print(f'random search wins {100*np.mean(np.array(rand_scores) > grid_best):.0f}% of the time')

**What to notice:** at an equal budget of 16 trials, **random search wins ~90%+ of the time**.
A 4×4 grid tries only **4 distinct `h1` values**, easily missing the narrow peak; random search
tries **16 distinct `h1` values**, sampling the dimension that matters four times as finely. This
is Bergstra & Bengio's classic result — random search's edge grows with the number of irrelevant
hyperparameters.

## 3 — Successive Halving: stop the losers early

Start many configs with a small budget, keep the top half, double their budget, repeat. Most configs
die cheap; compute concentrates on the survivors. We use noisy early estimates that improve with more
budget (more epochs = less noise).

In [ ]:
def noisy_eval(h1, h2, budget, seed):
    r = np.random.default_rng(seed)
    return val_score(h1, h2) + r.normal(0, 0.3 / np.sqrt(budget))   # noise shrinks with budget

def successive_halving(n_configs=16, min_budget=1):
    configs = [(rng.random(), rng.random(), i) for i in range(n_configs)]
    budget = min_budget; total = 0
    while len(configs) > 1:
        scored = [(noisy_eval(h1, h2, budget, sid), (h1, h2, sid)) for h1, h2, sid in configs]
        total += len(configs) * budget                           # compute spent this rung
        scored.sort(reverse=True)
        configs = [c for _, c in scored[:max(1, len(scored)//2)]]  # keep top half
        budget *= 2
    return configs[0], total

(best, tot) = successive_halving()
print(f'Successive Halving picked h1={best[0]:.2f} (target 0.5), total compute = {tot} budget-units')
print(f'Running all 16 configs to the max budget would cost {16 * 16} budget-units.')

**What to notice:** Successive Halving finds a good `h1` (near 0.5) for a **fraction** of the
compute of running all 16 configs to full budget. By evaluating everything cheaply first and only
doubling the budget of the surviving half each rung, it concentrates compute on promising configs —
the idea behind Hyperband and modern schedulers.

## 4. The library way — `sklearn`'s search CV

In practice you use `GridSearchCV` / `RandomizedSearchCV` (or Optuna, Ray Tune, Hyperband). The
cell tunes a `RandomForestClassifier` both ways at an equal 16-candidate budget and reports the
cross-validated best scores — the same grid-vs-random comparison, now on a real model.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

X, y = make_classification(n_samples=400, n_informative=6, random_state=0)
rf = RandomForestClassifier(n_estimators=40, random_state=0)

grid = GridSearchCV(rf, {'max_depth': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4, 8]},
                    cv=3).fit(X, y)
rand = RandomizedSearchCV(rf, {'max_depth': randint(2, 25), 'min_samples_leaf': randint(1, 10)},
                          n_iter=16, cv=3, random_state=0).fit(X, y)

print(f'GridSearchCV       best CV score = {grid.best_score_:.3f}  params={grid.best_params_}')
print(f'RandomizedSearchCV best CV score = {rand.best_score_:.3f}  params={rand.best_params_}')
assert 0.0 <= grid.best_score_ <= 1.0 and 0.0 <= rand.best_score_ <= 1.0
print('\nboth searches ran the same 16-candidate budget via sklearn ✓')

**What to notice:** both searches spend 16 candidates × 3 folds, but `RandomizedSearchCV`
samples `max_depth` and `min_samples_leaf` at many distinct values instead of a rigid 4×4 lattice —
the same advantage as the toy experiment, now with real cross-validation. For larger spaces you'd
reach for Optuna/Hyperband, which add the early-stopping idea from §3.

## 5. Gotchas & tradeoffs

- **Grid search scales exponentially.** `k` values across `d` hyperparameters = `kᵈ` trials —
  hopeless beyond a few dimensions. Random/Bayesian search scale far better.
- **Don't tune on the test set.** Hyperparameter search is *training*; use a validation split or
  cross-validation, and report on a held-out test set only once.
- **Successive Halving can drop slow-starters.** A config that's bad early but great later gets
  killed — the noise-vs-budget tradeoff (Hyperband hedges across bracket sizes).
- **More trials → more overfitting to the validation set.** Enough random configs will eventually
  fit the validation noise; nested CV guards against it.

In [ ]:
# Grid search's curse of dimensionality: k values per hyperparameter
k = 5
for d in [1, 2, 3, 5, 8]:
    print(f'{d} hyperparameters x {k} values each = {k**d:>6} grid trials')
print('\n-> random search covers each dimension with the SAME budget regardless of d')

**What to notice:** a 5-value grid over 8 hyperparameters is `5⁸ ≈ 390k` trials — completely
infeasible. Random search uses whatever fixed budget you give it no matter how many dimensions,
which is why it (and Bayesian/bandit methods) replaced grid search for anything beyond 2–3
hyperparameters.

## Key takeaways

- Hyperparameters are **searched**, not learned; **grid search wastes budget** on unimportant
  dimensions and scales as `kᵈ`.
- **Random search** samples important dimensions more finely and wins at equal budget (the
  Bergstra–Bengio result) — verified here on a toy surface and a real `RandomForest`.
- **Successive Halving / Hyperband** stop losers early to concentrate compute; **Bayesian
  optimization** models the surface.
- Always tune on **validation/CV**, never the test set; beware overfitting the validation set with
  too many trials.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/optimization-ml/06-quiz).

## ✏️ Your turn

**Exercise.** Implement `random_search(score_fn, budget, seed)` returning the best score over `budget`
random (h1, h2) draws in [0,1]², and `keep_top_half(scored)` returning the better half of a list of
`(score, config)` pairs (the core pruning step of Successive Halving).

In [ ]:
def random_search(score_fn, budget, seed=0):
    r = np.random.default_rng(seed)
    # TODO(you): draw `budget` random (h1,h2) in [0,1]^2, return the best score_fn value
    return ...

def keep_top_half(scored):
    # TODO(you): scored is a list of (score, config); return the top half by score (at least 1)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
s = random_search(val_score, 50, seed=1)
assert s <= best_possible + 1e-9                              # can't beat the true max
assert random_search(val_score, 200, 1) >= random_search(val_score, 4, 1)   # more budget, no worse
kept = keep_top_half([(0.1, 'a'), (0.9, 'b'), (0.5, 'c'), (0.3, 'd')])
assert set(c for _, c in kept) == {'b', 'c'}                  # the two highest scores
assert len(keep_top_half([(0.5, 'x')])) == 1                  # never drop the last one
# Edge case — a single trial (budget=1) must equal that one deterministic draw
r_check = np.random.default_rng(42)
expected_single = val_score(r_check.random(), r_check.random())
assert abs(random_search(val_score, 1, seed=42) - expected_single) < 1e-9, \
    "budget=1 should just be the score of that single random draw"

# Edge case — a tie in keep_top_half must still keep exactly half (rounded up)
tied = keep_top_half([(0.5, 'a'), (0.5, 'b')])
assert len(tied) == 1, "keep_top_half of 2 tied configs should keep exactly 1"

print('\u2713 random search and top-half pruning are correct')

<details>
<summary>Solution</summary>

```python
def random_search(score_fn, budget, seed=0):
    r = np.random.default_rng(seed)
    return max(score_fn(r.random(), r.random()) for _ in range(budget))

def keep_top_half(scored):
    scored = sorted(scored, reverse=True)
    return scored[:max(1, len(scored)//2)]
```

Random search wins because it samples the *important* hyperparameter at many distinct values instead
of wasting a grid on irrelevant ones; Successive Halving wins by not paying full price for configs
that are clearly losing.

</details>

---
## 🧪 Extra practice — learning rate schedule implementation bank

The [Learning Rate Schedules wiki page](https://ml-viz-ruby.vercel.app/wiki/learning-rate-schedules)
covers the full family (warmup, one-cycle, restarts, ...); here we implement three
of the classic decay schedules exactly as posed by
[DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem), each as a small
stateful scheduler class with an `__init__` and a `get_lr(epoch)` method:

- DML **153** — `StepLRScheduler` (drop by `gamma` every `step_size` epochs)
- DML **154** — `ExponentialLRScheduler` (drop by `gamma` every single epoch)
- DML **155** — `CosineAnnealingLRScheduler` (half-cosine curve down to `min_lr`
  over `T_max` epochs — the same formula as the `cosine_lr` function in the
  Gradient Descent Variants notebook, packaged as a class here instead of a
  plain function)

Fill in each `get_lr` body, then run the assert cell.

In [ ]:
# ── Schedule bank: StepLR (153), ExponentialLR (154), CosineAnnealingLR (155) ──
import math

class StepLRScheduler:
    """DML 153. lr drops by gamma every step_size epochs — a staircase decay."""
    def __init__(self, initial_lr, step_size, gamma):
        self.initial_lr = initial_lr
        self.step_size = step_size
        self.gamma = gamma

    def get_lr(self, epoch):
        # TODO(you): return round(initial_lr * gamma ** (epoch // step_size), 4)
        return ...


class ExponentialLRScheduler:
    """DML 154. lr drops by gamma every single epoch — a smooth exponential decay."""
    def __init__(self, initial_lr, gamma):
        self.initial_lr = initial_lr
        self.gamma = gamma

    def get_lr(self, epoch):
        # TODO(you): return round(initial_lr * gamma ** epoch, 4)
        return ...


class CosineAnnealingLRScheduler:
    """DML 155. lr follows a half-cosine curve from initial_lr down to min_lr
    over T_max epochs."""
    def __init__(self, initial_lr, T_max, min_lr):
        self.initial_lr = initial_lr
        self.T_max = T_max
        self.min_lr = min_lr

    def get_lr(self, epoch):
        # TODO(you): lr = min_lr + 0.5*(initial_lr - min_lr)*(1 + cos(pi*epoch/T_max))
        # TODO(you): return round(lr, 4)
        return ...

In [ ]:
# Checks — run me after filling in the three schedulers above

# 1) Single-step correctness against DML-OpenProblem reference values
step_sched = StepLRScheduler(initial_lr=0.1, step_size=5, gamma=0.5)
assert [step_sched.get_lr(e) for e in (0, 4, 5, 9, 10)] == [0.1, 0.1, 0.05, 0.05, 0.025], \
    "StepLR values don't match the DML reference"

exp_sched = ExponentialLRScheduler(initial_lr=0.1, gamma=0.9)
assert [exp_sched.get_lr(e) for e in (0, 1, 2, 3)] == [0.1, 0.09, 0.081, 0.0729], \
    "ExponentialLR values don't match the DML reference"

cos_sched = CosineAnnealingLRScheduler(initial_lr=0.1, T_max=10, min_lr=0.001)
assert [cos_sched.get_lr(e) for e in (0, 2, 5, 7, 10)] == [0.1, 0.0905, 0.0505, 0.0214, 0.001], \
    "CosineAnnealingLR values don't match the DML reference"

# 2) Edge case — schedule at step 0 and at the final step (T_max)
assert cos_sched.get_lr(0) == cos_sched.initial_lr, "epoch 0 should equal initial_lr"
assert cos_sched.get_lr(10) == cos_sched.min_lr, "epoch T_max should equal min_lr"
step_final = StepLRScheduler(initial_lr=0.001, step_size=50, gamma=0.5)
assert step_final.get_lr(0) == 0.001, "StepLR at epoch 0 should equal initial_lr"
assert step_final.get_lr(100) == 0.0003, "StepLR after two full decay periods"

# 3) Edge case — a single-epoch schedule (T_max=1): epoch 0 is initial_lr,
#    epoch 1 (=T_max, the only other step) is min_lr
single_step_cos = CosineAnnealingLRScheduler(initial_lr=0.001, T_max=1, min_lr=0.0001)
assert single_step_cos.get_lr(0) == 0.001
assert single_step_cos.get_lr(1) == 0.0001

# 4) Edge case — a learning rate of 0 stays 0 for every schedule, at every epoch
assert StepLRScheduler(0.0, 5, 0.5).get_lr(20) == 0.0
assert ExponentialLRScheduler(0.0, 0.9).get_lr(100) == 0.0
assert CosineAnnealingLRScheduler(0.0, 10, 0.0).get_lr(5) == 0.0

print("✅ Schedule bank passed")

<details>
<summary>Solution</summary>

```python
import math

class StepLRScheduler:
    def __init__(self, initial_lr, step_size, gamma):
        self.initial_lr = initial_lr
        self.step_size = step_size
        self.gamma = gamma

    def get_lr(self, epoch):
        return round(self.initial_lr * self.gamma ** (epoch // self.step_size), 4)

class ExponentialLRScheduler:
    def __init__(self, initial_lr, gamma):
        self.initial_lr = initial_lr
        self.gamma = gamma

    def get_lr(self, epoch):
        return round(self.initial_lr * self.gamma ** epoch, 4)

class CosineAnnealingLRScheduler:
    def __init__(self, initial_lr, T_max, min_lr):
        self.initial_lr = initial_lr
        self.T_max = T_max
        self.min_lr = min_lr

    def get_lr(self, epoch):
        lr = self.min_lr + 0.5 * (self.initial_lr - self.min_lr) * (
            1 + math.cos(math.pi * epoch / self.T_max)
        )
        return round(lr, 4)
```

StepLR and ExponentialLR are both geometric decays — StepLR just holds the rate
constant between drops instead of decaying every epoch, which is why it needs
`step_size` as well as `gamma`. Cosine annealing is the odd one out: instead of
approaching 0 asymptotically, it reaches an exact floor (`min_lr`) at `T_max`
and would (if you kept calling `get_lr` past `T_max`) start climbing back up —
that's the basis of "warm restarts" (SGDR), covered on the wiki page.

</details>